##  (1) Data Loading

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_eng = pd.read_csv(r'D:\Neurova_AI\Language-Specific_Sentiment_Analysis\data\MovieReviewTrainingDatabase.csv', encoding='latin-1')
df_eng.head()

,sentiment,review
0,Positive,With all this stuff going down at the moment w...
1,Positive,'The Classic War of the Worlds' by Timothy Hin...
2,Negative,The film starts with a manager (Nicholas Bell)...
3,Negative,It must be assumed that those who praised this...
4,Positive,Superbly trashy and wondrously unpretentious 8...


In [3]:
df_eng.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   sentiment  25000 non-null  str  
 1   review     25000 non-null  str  
dtypes: str(2)
memory usage: 31.7 MB


In [6]:
df_eng['sentiment'].unique()

<ArrowStringArray>
['Positive', 'Negative']
Length: 2, dtype: str

## (2) Data Preprocessing

In [7]:
# 2.1 : remove the stop words 
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

# for sentiment analysis, negative words has an impact on the meaning so it can't be removed blindly
# so i will remove the stop words execluding from them the negative words

negation_words = {
    'no', 'not', 'nor', 'never', "don't", "doesn't",
    "didn't", "isn't", "wasn't", "weren't", "won't",
    "wouldn't", "can't", "couldn't", "shouldn't"
}

stop_words = stop_words - negation_words 

In [8]:
# 2.2 : remove the punctuation and special characters using regex
import re

#this is a ready function to clean the text data
def preprocess_english(text):
    text = text.lower()

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)

    words = text.split()

    words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(words)

df_eng['review'] = df_eng['review'].apply(preprocess_english)



In [9]:
df_eng.head()

,sentiment,review
0,Positive,"stuff going moment mj started listening music,..."
1,Positive,'the classic war worlds' timothy hines enterta...
2,Negative,film starts manager (nicholas bell) giving wel...
3,Negative,must assumed praised film ('the greatest filme...
4,Positive,superbly trashy wondrously unpretentious 80's ...


In [12]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sch\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sch\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sch\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [15]:
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import wordnet
def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1]

    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\sch\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


In [16]:
# 2.3: text tokenization and lemmatization
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

tokenized_reviews = df_eng['review'].apply(word_tokenize)
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    tokens = word_tokenize(text)

    lemmatized_tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(word))
        for word in tokens
    ]
    return " ".join(lemmatized_tokens)

df_eng['review'] = df_eng['review'].apply(lemmatize_text)
df_eng.head()

,sentiment,review
0,Positive,"stuff go moment mj start listen music , watch ..."
1,Positive,'the classic war world ' timothy hines enterta...
2,Negative,film start manager ( nicholas bell ) give welc...
3,Negative,must assume praise film ( 'the great film oper...
4,Positive,superbly trashy wondrously unpretentious 80 's...
